Q1. Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)

A) 30 ngày
B) 90 ngày
C) 144 ngày
D) 365 ngày

In [ ]:
import pandas as pd
from pathlib import Path

base = next((p for p in [Path('dataset/round1'), Path('Task1/dataset/round1')] if p.exists()), None)
if base is None:
    raise FileNotFoundError('Khong tim thay dataset/round1')

orders = pd.read_csv(base / 'orders.csv', parse_dates=['order_date'], low_memory=False)
order_items = pd.read_csv(base / 'order_items.csv', low_memory=False)
products = pd.read_csv(base / 'products.csv', low_memory=False)
returns = pd.read_csv(base / 'returns.csv', parse_dates=['return_date'], low_memory=False)
web_traffic = pd.read_csv(base / 'web_traffic.csv', parse_dates=['date'], low_memory=False)
customers = pd.read_csv(base / 'customers.csv', parse_dates=['signup_date'], low_memory=False)
geography = pd.read_csv(base / 'geography.csv', low_memory=False)
payments = pd.read_csv(base / 'payments.csv', low_memory=False)
sales = pd.read_csv(base / 'sales.csv', parse_dates=['Date'], low_memory=False)

gaps = (
    orders.sort_values(['customer_id', 'order_date'])
    .groupby('customer_id')['order_date']
    .diff()
    .dt.days
    .dropna()
)
median_gap = float(gaps.median())
options = {'A': 30, 'B': 90, 'C': 144, 'D': 365}
closest_choice = min(options, key=lambda k: abs(options[k] - median_gap))

print('Q1 - inter-order gap')
print(f'Tong so gap: {len(gaps):,}')
print(f'Median gap: {median_gap:.2f} ngay')
print(f'25% / 75% quantile: {gaps.quantile(0.25):.1f} / {gaps.quantile(0.75):.1f} ngay')
print(f'Dap an gan nhat theo option: {closest_choice} ({options[closest_choice]} ngay)')


Q1 - inter-order gap
Tong so gap: 556,699
Median gap: 144.00 ngay
25% / 75% quantile: 46.0 / 357.0 ngay
Dap an gan nhat theo option: C (180 ngay)


Q2. Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức (price - cogs)/price?

A) Premium
B) Performance
C) Activewear
D) Standard

In [2]:
summary = (
    products.assign(gross_margin=(products['price'] - products['cogs']) / products['price'])
    .groupby('segment', as_index=False)['gross_margin']
    .mean()
    .sort_values('gross_margin', ascending=False)
)

print('Q2 - average gross margin by segment')
print(summary.to_string(index=False, formatters={'gross_margin': '{:.4f}'.format}))
top = summary.iloc[0]
print(f"Top segment: {top['segment']} with mean gross margin {top['gross_margin']:.4f}")


Q2 - average gross margin by segment
    segment gross_margin
   Standard       0.3134
    Premium       0.2854
All-weather       0.2842
 Activewear       0.2656
Performance       0.2636
   Balanced       0.2580
     Trendy       0.2408
   Everyday       0.2363
Top segment: Standard with mean gross margin 0.3134


Q3. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

A) defective
B) wrong_size
C) changed_mind
D) not_as_described

In [3]:
streetwear_returns = returns.merge(products[['product_id', 'category']], on='product_id', how='inner')
streetwear_returns = streetwear_returns[streetwear_returns['category'].eq('Streetwear')]
reason_counts = streetwear_returns['return_reason'].value_counts()

print('Q3 - Streetwear return reasons')
print(f'Tong so return record Streetwear: {len(streetwear_returns):,}')
print(reason_counts.head(5).to_string())
if not reason_counts.empty:
    print(f"Top reason: {reason_counts.index[0]} ({reason_counts.iloc[0]:,} records)")


Q3 - Streetwear return reasons
Tong so return record Streetwear: 21,799
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Top reason: wrong_size (7,626 records)


Q4. Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

A) organic_search
B) paid_search
C) email_campaign
D) social_media

In [4]:
traffic_summary = (
    web_traffic.groupby('traffic_source', as_index=False)['bounce_rate']
    .mean()
    .sort_values('bounce_rate', ascending=True)
)

print('Q4 - mean bounce rate by traffic source')
print(traffic_summary.to_string(index=False, formatters={'bounce_rate': '{:.5f}'.format}))
top = traffic_summary.iloc[0]
print(f"Lowest bounce_rate source: {top['traffic_source']} ({top['bounce_rate']:.5f})")


Q4 - mean bounce rate by traffic source
traffic_source bounce_rate
email_campaign     0.00446
  social_media     0.00448
   paid_search     0.00448
      referral     0.00450
organic_search     0.00450
        direct     0.00451
Lowest bounce_rate source: email_campaign (0.00446)


Q5. Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id không null) xấp xỉ là bao nhiêu?

A) 12%
B) 25%
C) 39%
D) 54%

In [5]:
promo_mask = order_items['promo_id'].notna()
promo_share = promo_mask.mean() * 100

print('Q5 - promo usage in order_items')
print(f'Tong dong: {len(order_items):,}')
print(f'Dong co promo_id khong null: {promo_mask.sum():,}')
print(f'Ty le co khuyen mai: {promo_share:.2f}%')


Q5 - promo usage in order_items
Tong dong: 714,669
Dong co promo_id khong null: 276,316
Ty le co khuyen mai: 38.66%


Q6. Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong nhóm)

A) 55+
B) 25-34
C) 35-44
D) 45-54

In [6]:
valid_customers = customers[customers['age_group'].notna()][['customer_id', 'age_group']]
customer_orders = orders.merge(valid_customers, on='customer_id', how='inner')
age_summary = (
    customer_orders.groupby('age_group')
    .agg(total_orders=('order_id', 'size'), customers=('customer_id', 'nunique'))
    .reset_index()
)
age_summary['avg_orders_per_customer'] = age_summary['total_orders'] / age_summary['customers']
age_summary = age_summary.sort_values('avg_orders_per_customer', ascending=False)

print('Q6 - orders per customer by age group')
print(age_summary.to_string(index=False, formatters={'avg_orders_per_customer': '{:.4f}'.format}))
top = age_summary.iloc[0]
print(f"Top age group: {top['age_group']} ({top['avg_orders_per_customer']:.4f} orders/customer)")


Q6 - orders per customer by age group
age_group  total_orders  customers avg_orders_per_customer
      55+         72760      10010                  7.2687
    45-54        124138      17193                  7.2203
    35-44        170368      23642                  7.2062
    25-34        190622      26802                  7.1122
    18-24         89057      12599                  7.0686
Top age group: 55+ (7.2687 orders/customer)


Q7. Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong sales_train.csv?

A) West
B) Central
C) East
D) Cả ba vùng có doanh thu xấp xỉ bằng nhau

In [7]:
region_revenue = (
    orders[['order_id', 'zip']]
    .merge(order_items[['order_id', 'quantity', 'unit_price']], on='order_id', how='inner')
    .merge(geography[['zip', 'region']], on='zip', how='inner')
)
region_revenue['line_revenue'] = region_revenue['quantity'] * region_revenue['unit_price']
region_summary = (
    region_revenue.groupby('region', as_index=False)['line_revenue']
    .sum()
    .sort_values('line_revenue', ascending=False)
)

print('Q7 - total revenue by region (based on order_items line revenue)')
print(region_summary.to_string(index=False, formatters={'line_revenue': '{:,.2f}'.format}))
top = region_summary.iloc[0]
print(f"Top region: {top['region']} with revenue {top['line_revenue']:,.2f}")


Q7 - total revenue by region (based on order_items line revenue)
 region     line_revenue
   East 7,637,532,676.20
Central 4,941,908,471.68
   West 3,851,035,437.65
Top region: East with revenue 7,637,532,676.20


Q8. Trong các đơn hàng có order_status = 'cancelled' trong orders.csv, phương thức thanh toán nào được sử dụng nhiều nhất?

A) credit_card
B) cod
C) paypal
D) bank_transfer

In [8]:
cancelled = orders[orders['order_status'].eq('cancelled')]
payment_counts = cancelled['payment_method'].value_counts()

print('Q8 - payment methods among cancelled orders')
print(f'Tong so cancelled orders: {len(cancelled):,}')
print(payment_counts.to_string())
if not payment_counts.empty:
    print(f"Top payment method: {payment_counts.index[0]} ({payment_counts.iloc[0]:,} orders)")


Q8 - payment methods among cancelled orders
Tong so cancelled orders: 59,462
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Top payment method: credit_card (28,452 orders)


Q9. Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join với products theo product_id)?

A) S
B) M
C) L
D) XL

In [9]:
sizes = ['S', 'M', 'L', 'XL']
order_size = order_items.merge(products[['product_id', 'size']], on='product_id', how='left')
return_size = returns.merge(products[['product_id', 'size']], on='product_id', how='left')

summary = pd.DataFrame({'size': sizes})
summary['order_item_rows'] = summary['size'].map(order_size.groupby('size').size()).fillna(0).astype(int)
summary['return_rows'] = summary['size'].map(return_size.groupby('size').size()).fillna(0).astype(int)
summary['return_rate'] = summary['return_rows'] / summary['order_item_rows']
summary = summary.sort_values('return_rate', ascending=False)

print('Q9 - return rate by size')
print(summary.to_string(index=False, formatters={'return_rate': '{:.6f}'.format}))
top = summary.iloc[0]
print(f"Top size: {top['size']} with return rate {top['return_rate']:.6f}")


Q9 - return rate by size
size  order_item_rows  return_rows return_rate
   S           172042         9723    0.056515
   L           173174         9741    0.056250
   M           176428         9820    0.055660
  XL           193025        10655    0.055200
Top size: S with return rate 0.056515


Q10. Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?

A) 1 kỳ (trả một lần)
B) 3 kỳ
C) 6 kỳ
D) 12 kỳ

In [10]:
installment_summary = (
    payments.groupby('installments', as_index=False)
    .agg(mean_payment_value=('payment_value', 'mean'), order_count=('payment_value', 'size'))
    .sort_values('mean_payment_value', ascending=False)
)

print('Q10 - average payment value by installment count')
print(installment_summary.to_string(index=False, formatters={'mean_payment_value': '{:,.2f}'.format}))
top = installment_summary.iloc[0]
print(f"Top plan: {int(top['installments'])} installments with mean payment {top['mean_payment_value']:,.2f}")


Q10 - average payment value by installment count
 installments mean_payment_value  order_count
            6          24,446.65       109910
            3          24,399.64       218949
           12          24,245.77        54126
            1          24,113.27       262866
            2             708.47         1094
Top plan: 6 installments with mean payment 24,446.65
